In [1]:
import requests
from dotenv import load_dotenv
import os
import pandas as pd
import rasterio
import numpy as np
import glob
import geopandas as gpd
from rasterstats import zonal_stats

load_dotenv('../envs/dev.env')

token = os.getenv("HCDP_ADMIN_TOKEN")
local_dep_dir = os.environ.get('DEPENDENCY_DIR')

headers = {
    "Authorization": f"Bearer {token}"
}

In [133]:
# date, startDate, endDate, island, division_type, and name. For  island, division_type, and name you can provide multiple instances of each or a comma separated list to filter by more than one (e.g. ?division_type=moku&division_type=island or  ?division_type=moku,island)
#Viewing data

url = "https://api.hcdp.ikewai.org/mesonet/climate_report/rainfall_stats"
# rainfall_stats, temperature_stats, drought_stats, rainfall_historical, or temperature_historical
query_params = {
    "division_type": "Statewide",
    "startDate": "1920-03-01",
    "endDate": "2026-03-01"
}

response = requests.get(url, params=query_params, headers=headers)

print(f"Status Code: {response.status_code}")
# print(response.json())
if response.status_code == 200:
    df_filtered = pd.DataFrame(response.json())
    display(df_filtered)
else:
    print(f"Response Text: {response.text}")

Status Code: 200


,island,division_type,name,date,mean,anomaly,pchange,rank,ytd_pnormal
0,Statewide,Statewide,Statewide,1958-01-01T00:00:00.000Z,2.4,-2.8,-54,91,46
1,Statewide,Statewide,Statewide,1958-02-01T00:00:00.000Z,3.4,-1.7,-34,76,56
2,Statewide,Statewide,Statewide,1958-03-01T00:00:00.000Z,9.6,2.7,40,30,90
3,Statewide,Statewide,Statewide,1958-04-01T00:00:00.000Z,2.9,-2.1,-43,99,82
4,Statewide,Statewide,Statewide,1958-05-01T00:00:00.000Z,4.7,0.6,16,48,87
...,...,...,...,...,...,...,...,...,...
1268,Statewide,Statewide,Statewide,1981-08-01T00:00:00.000Z,6.4,1.2,24,25,74
1269,Statewide,Statewide,Statewide,1981-09-01T00:00:00.000Z,4.6,0.2,4,34,77
1270,Statewide,Statewide,Statewide,1981-10-01T00:00:00.000Z,5,-0.1,-3,49,79
1271,Statewide,Statewide,Statewide,1981-11-01T00:00:00.000Z,7.5,1,16,33,83


In [135]:
df_filtered['date'] = pd.to_datetime(df_filtered['date'])
df_filtered['rank'] = df_filtered['rank'].astype(int)
# 2. Filter the DataFrame to keep only rows where the month is 8 (August)
august_df = df_filtered[df_filtered['date'].dt.month == 3]

# View the result
august_df

,island,division_type,name,date,mean,anomaly,pchange,rank,ytd_pnormal
2,Statewide,Statewide,Statewide,1958-03-01 00:00:00+00:00,9.6,2.7,40,30,90
10,Statewide,Statewide,Statewide,1992-03-01 00:00:00+00:00,2.3,-4.6,-67,101,39
21,Statewide,Statewide,Statewide,1993-03-01 00:00:00+00:00,3.7,-3.2,-46,83,50
33,Statewide,Statewide,Statewide,1994-03-01 00:00:00+00:00,10.1,3.3,48,25,151
45,Statewide,Statewide,Statewide,1997-03-01 00:00:00+00:00,10.8,3.9,57,19,130
...,...,...,...,...,...,...,...,...,...
1206,Statewide,Statewide,Statewide,1977-03-01 00:00:00+00:00,9.9,3,44,27,89
1218,Statewide,Statewide,Statewide,1978-03-01 00:00:00+00:00,5.5,-1.4,-20,69,57
1242,Statewide,Statewide,Statewide,1984-03-01 00:00:00+00:00,2.5,-4.3,-63,99,68
1255,Statewide,Statewide,Statewide,1980-03-01 00:00:00+00:00,24.6,17.8,259,1,221


In [118]:
# date, startDate, endDate, island, division_type, and name. For  island, division_type, and name you can provide multiple instances of each or a comma separated list to filter by more than one (e.g. ?division_type=moku&division_type=island or  ?division_type=moku,island)
#Viewing data

url = "https://api.hcdp.ikewai.org/mesonet/climate_report/rainfall_stats"
# rainfall_stats, temperature_stats, drought_stats, rainfall_historical, or temperature_historical
#islands: Moloka‘i, Lāna‘i
query_params = {
    "division_type": "ahupuaa",
    "name":"Ahalanui\\, Laepaoʻo",
    # "name": "Haʻakoa",
    # "date":"2026-03"
}

response = requests.get(url, params=query_params, headers=headers)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    df_filtered = pd.DataFrame(response.json())
    display(df_filtered)
else:
    print(f"Response Text: {response.text}")

Status Code: 200


,island,division_type,name,date,mean,anomaly,pchange,rank,ytd_pnormal
0,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1941-07-01T00:00:00.000Z,4.3,-2.9,-40,88,64
1,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1967-06-01T00:00:00.000Z,5.5,-0.1,-2.5,35,146
2,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1968-04-01T00:00:00.000Z,15.7,8.7,123.2,7,168
3,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1967-10-01T00:00:00.000Z,6.7,-1.9,-22.4,59,132
4,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1945-01-01T00:00:00.000Z,3.2,-3.9,-54.7,85,45
...,...,...,...,...,...,...,...,...,...
1270,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1967-12-01T00:00:00.000Z,8.9,-2,-18.7,56,127
1271,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1968-01-01T00:00:00.000Z,11,3.9,55.8,37,156
1272,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1968-03-01T00:00:00.000Z,12.9,3.9,43.9,36,152
1273,Hawaiʻi,ahupuaa,"Ahalanui, Laepaoʻo",1994-05-01T00:00:00.000Z,5.3,-0.4,-7.7,56,138


In [3]:
def convert_units(value, dataset):
  """Convert rainfall mm to inches and temperature C to F"""
  if value is None or np.isnan(value):
      return np.nan
  if dataset == "rainfall":
      return value / 25.4
  elif dataset == "temperature":
      return (value * 9/5) + 32
  return value

def make_ytd(year, month):
  input_dir = os.path.join("/Users/cherryleheu/Documents/HCDP/Data/rf_all/monthly/")
  output_dir = os.path.join("/Users/cherryleheu/Documents/HCDP/Data/rf_all/YTD/")
  first_file = os.path.join(input_dir, f"rainfall_{year}_01.tif")

  if not os.path.exists(first_file):
      print(f"Warning: Missing starting file {first_file} for YTD calculation.")
      return None

  with rasterio.open(first_file) as src:
      meta = src.meta.copy()
      first_data = src.read(1, masked=True)
      land_mask = first_data.mask
      ytd_sum = np.zeros(first_data.shape, dtype='float32')

  for m in range(1, month + 1):
      file_path = os.path.join(input_dir, f"rainfall_{year}_{m:02d}.tif")
      if os.path.exists(file_path):
          with rasterio.open(file_path) as src:
              data = src.read(1).astype('float32')
              if src.nodata is not None:
                  data[data == src.nodata] = 0
              ytd_sum += data

  ytd_sum[land_mask] = -9999
  meta.update(dtype='float32', nodata=-9999)
  output_path = os.path.join(output_dir, f'YTD_{year}_{month:02d}.tif')

  with rasterio.open(output_path, 'w', **meta) as dst:
      dst.write(ytd_sum, 1)

  return output_path

def get_statewide_stats(dataset, year, month):
  """Compute statewide mean, anomaly, percent change, and drought percentage."""
  climo_file = os.path.join(local_dep_dir, f"climo/{dataset}/{dataset}_1991-2020_{month:02d}.tif")

  print(f"\n--- Processing statewide ({dataset}) ---")

  with rasterio.open(climo_file) as src:
      clim = src.read(1).astype(float)
      clim = np.where(src.nodata == src.read(1), np.nan, clim)
      climo_mean = convert_units(np.nanmean(clim), dataset)

  all_records = []

  if dataset == "rainfall":
    tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/monthly/"
    var = "rainfall"
  elif dataset == "temperature":
    tif_path = "/Users/cherryleheu/Documents/HCDP/Data/monthly/tmean/"
    var = "tmean"

  for tif in sorted(glob.glob(os.path.join(tif_path, f"{var}_*_{month:02d}.tif"))):
      parts = os.path.basename(tif).replace(".tif", "").split("_")
      curr_year, curr_month = parts[1], parts[2]
      curr_date = f"{curr_year}-{curr_month}"

      with rasterio.open(tif) as src:
          arr = src.read(1).astype(float)
          arr = np.where(arr == src.nodata, np.nan, arr)
          mean_val = convert_units(np.nanmean(arr), dataset)

          if dataset == "temperature":
              with np.errstate(all='ignore'):
                  max_val = convert_units(np.nanmax(arr), dataset)

      anomaly = mean_val - climo_mean
      pchange = ((mean_val - climo_mean) / climo_mean) * 100 if dataset == "rainfall" else anomaly

      record = {
          "date": curr_date,
          "mean": round(mean_val, 1),
          "anomaly": round(anomaly, 1),
          "pchange": int(round(pchange, 0)),
      }
      if dataset == "temperature":
          record["max"] = int(round(max_val, 0))

      all_records.append(record)

  df = pd.DataFrame(all_records)

  df["rank"] = df["anomaly"].rank(method="min", ascending=False)
  latest = df[df["date"] == f"{year}-{month:02d}"].copy()
  if latest.empty:
      print(f"No data found for {year}-{month:02d}")
      return

  if dataset == "rainfall":
      current_ytd_path = make_ytd(year, month)
      climo_ytd_path = os.path.join(local_dep_dir, "climo", "rainfall_ytd", f"YTD_rain_month_{month:02d}.tif")

      with rasterio.open(current_ytd_path) as src:
          curr_arr = src.read(1).astype(float)
          curr_arr = np.where(curr_arr == -9999, np.nan, curr_arr)
          curr_mean = np.nanmean(curr_arr)

      with rasterio.open(climo_ytd_path) as src:
          climo_arr = src.read(1).astype(float)
          climo_arr = np.where(climo_arr == -9999, np.nan, climo_arr)
          climo_mean = np.nanmean(climo_arr)

      if not np.isnan(curr_mean) and not np.isnan(climo_mean) and climo_mean != 0:
          pnormal = (curr_mean / climo_mean) * 100
      else:
          pnormal = np.nan

      latest["ytd_pnormal"] = int(round(pnormal, 0))

  latest["island"] = "Statewide"
  latest["division_type"] = "Statewide"
  latest["name"] = "Statewide"

  base_cols = ["island", "division_type", "name", "date", "mean", "anomaly", "pchange", "rank"]

  if dataset == "rainfall":
      base_cols.append("ytd_pnormal")
  elif dataset == "temperature":
      base_cols.append("max")

  result_list = [x.item() if hasattr(x, 'item') else x for x in latest[base_cols].iloc[0].tolist()]

  print(f"Returning stats for {year}-{month:02d}: {result_list}")
  return result_list


In [4]:
def get_stats(division, dataset, year, month):
  """Compute rainfall or temperature statistics for a island or division shapefile."""
  print(f"\n--- Processing {division} - {dataset} ---")

  shapefile = os.path.join(local_dep_dir, f"shapefiles/{division}.shp")
  climo_file = os.path.join(local_dep_dir, f"climo/{dataset}/{dataset}_1991-2020_{month:02d}.tif")

  gdf = gpd.read_file(shapefile).copy()

  island_col = next((c for c in gdf.columns if c.lower() in ["island", "mokupuni", "isle", "islandname"]), None)
  name_col = next((c for c in gdf.columns if c.lower() in ["name", "division", "moku", "climate_div", "ahupuaa", "county", "name_hwn"]), None)

  gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.001, preserve_topology=True)

  if island_col and name_col:
      # Standard logic for complex shapefiles (moku, ahupuaa)
      is_same_island_dup = gdf.duplicated(subset=[island_col, name_col], keep=False)
      cum_count = gdf.groupby([island_col, name_col]).cumcount() + 1
      gdf.loc[is_same_island_dup, name_col] = (
          gdf.loc[is_same_island_dup, name_col].astype(str) + " " + cum_count[is_same_island_dup].astype(str)
      )
      gdf = gdf.dissolve(by=[island_col, name_col], as_index=False)
      gdf["island_clean"] = gdf[island_col]
      gdf["name_clean"] = gdf[name_col]
  elif name_col:
      gdf = gdf.dissolve(by=name_col, as_index=False)
      gdf["name_clean"] = gdf[name_col]

      if division == "island":
          gdf["island_clean"] = gdf[name_col]
      else:
          gdf["island_clean"] = "Statewide"

  gdf["division_type"] = division

  climo_zs = zonal_stats(vectors=gdf, raster=climo_file, stats=["mean"], nodata=None)
  gdf["climo_mean"] = [convert_units(c["mean"], dataset) for c in climo_zs]

  all_records = []

  if dataset == "rainfall":
      tif_path = "/Users/cherryleheu/Documents/HCDP/Data/rf_all/monthly/"
      var = "rainfall"
  elif dataset == "temperature":
      tif_path = "/Users/cherryleheu/Documents/HCDP/Data/monthly/tmean/"
      var = "tmean"

  for tif in sorted(glob.glob(os.path.join(tif_path, f"{var}_*_{month:02d}.tif"))):
      parts = os.path.basename(tif).replace(".tif", "").split("_")
      curr_year, curr_month = parts[1], parts[2]
      curr_date = f"{curr_year}-{curr_month}"

      stats_to_compute = ["mean", "max"] if dataset == "temperature" else ["mean"]
      stats = zonal_stats(vectors=gdf, raster=tif, stats=stats_to_compute, nodata=None)

      for idx, row in gdf.iterrows():
          mean_raw = stats[idx]["mean"]

          if mean_raw is None or np.isnan(mean_raw):
              mean_val, anomaly, pchange = np.nan, np.nan, np.nan
              if dataset == "temperature":
                  max_val = np.nan
          else:
              mean_val = convert_units(mean_raw, dataset)
              climo_mean = row["climo_mean"]
              if np.isnan(climo_mean):
                  anomaly, pchange = np.nan, np.nan
              else:
                  anomaly = mean_val - climo_mean
                  pchange = ((mean_val - climo_mean) / climo_mean) * 100 if dataset == "rainfall" else anomaly

              if dataset == "temperature":
                  max_raw = stats[idx].get("max")
                  max_val = convert_units(max_raw, dataset) if max_raw is not None else np.nan

          record = {
              "island": row["island_clean"],
              "division_type": row["division_type"],
              "name": row["name_clean"],
              "date": curr_date,
              "mean": round(mean_val, 1),
              "anomaly": round(anomaly, 1),
              "pchange": round(pchange, 1),
          }
          if dataset == "temperature":
              # FIXED: Check if max_val is NaN before converting to int
              record["max"] = int(round(max_val, 0)) if not np.isnan(max_val) else np.nan
          all_records.append(record)

  df = pd.DataFrame(all_records)

  df["rank"] = df.groupby(["island", "name"])["anomaly"].rank(method="min", ascending=False)
  latest_df = df[df["date"] == f"{year}-{month:02d}"].reset_index(drop=True)

  if dataset == "rainfall":
      current_ytd_path = make_ytd(year, month)
      climo_ytd_path = os.path.join(local_dep_dir, "climo", "rainfall_ytd", f"YTD_rain_month_{month:02d}.tif")

      current_ytd_zs = zonal_stats(vectors=gdf, raster=current_ytd_path, stats=["mean"], nodata=-9999)
      climo_ytd_zs = zonal_stats(vectors=gdf, raster=climo_ytd_path, stats=["mean"], nodata=-9999)

      ytd_pnormals = []
      for curr, climo in zip(current_ytd_zs, climo_ytd_zs):
          curr_mean = curr['mean']
          climo_mean = climo['mean']

          if curr_mean is not None and climo_mean is not None and climo_mean != 0:
              pnormal = (curr_mean / climo_mean) * 100
          else:
              pnormal = np.nan

          ytd_pnormals.append(pnormal)
              
      # FIXED: Check if the ytd_pnormals value is NaN before converting to int
      latest_df["ytd_pnormal"] = int(round(ytd_pnormals[0], 0)) if ytd_pnormals and not np.isnan(ytd_pnormals[0]) else np.nan

  base_cols = ["island", "division_type", "name", "date", "mean", "anomaly", "pchange", "rank"]

  if dataset == "rainfall":
      base_cols.append("ytd_pnormal")
  elif dataset == "temperature":
      base_cols.append("max")

  final_data = []
  for _, row in latest_df[base_cols].iterrows():
      # Convert each row to native Python types to avoid JSON errors
      row_list = [
          None if (isinstance(x, float) and np.isnan(x)) # Handle NaNs
          else (x.item() if hasattr(x, 'item') else x)   # Convert NumPy to Python
          for x in row
      ]
      final_data.append(row_list)

  print(f"Returning stats for {len(final_data)} locations for {year}-{month:02d}")
  return final_data

In [7]:
data_list

NameError: name 'data_list' is not defined

In [5]:
import requests
from concurrent.futures import ThreadPoolExecutor

# Configuration
dataset = "rainfall"
url = f"https://api.hcdp.ikewai.org/mesonet/climate_report/{dataset}_stats"
start_year = 1920
end_year = 1920
current_month_limit = 3
max_threads = 5  # Number of years to process at once

def upload_year(year):
    """Function to process and post a single year."""
    data_list = []
    last_month = current_month_limit if year == end_year else 12
    
    # Collect months
    for month in range(1, last_month + 1):
        # Ensure get_stats is defined in your notebook
        stats = get_stats("ahupuaa", dataset, year, month)
        data_list.append(stats)

    payload = {
        "overwrite": True,
        "data": data_list
    }

    try:
        # headers must be defined in a previous cell or globally
        response = requests.post(url, json=payload, headers=headers)
        return f"Year {year}: {response.status_code}"
    except Exception as e:
        return f"Year {year}: Error - {e}"

# Run the threading pool
years = range(start_year, end_year + 1)

with ThreadPoolExecutor(max_workers=max_threads) as executor:
    # This maps the function to the list of years and returns results as they finish
    results = list(executor.map(upload_year, years))

# Print the results summary
for res in results:
    print(res)


--- Processing ahupuaa - rainfall ---
Returning stats for 726 locations for 1920-01

--- Processing ahupuaa - rainfall ---
Returning stats for 726 locations for 1920-02

--- Processing ahupuaa - rainfall ---


KeyboardInterrupt: 

Returning stats for 726 locations for 1920-03

In [23]:
payload = {
    "overwrite": True,
    "data": data_list[0]
}

# Send the request
response = requests.post(url, json=payload, headers=headers)
print(f"Status Code: {response.status_code}")

Status Code: 200


In [ ]:

url = "https://api.hcdp.ikewai.org/mesonet/climate_report/rainfall_stats"

# const STAT_TABLE_DATA = {
#   rainfall_stats: ['Statewide', 'Statewide', 'Statewide', 'date', 'mean', 'anomaly', 'pchange', 'rank', 'ytd_pnormal'],
#   temperature_stats: ['Statewide', 'Statewide', 'Statewide', 'date', 'mean', 'anomaly', 'pchange', 'rank', 'max'],
#   drought_stats: ['division_full', 'date', 'd4', 'd3', 'd2', 'd1', 'd0', 'near_normal', 'w0', 'w1', 'w2', 'w3', 'w4'],
#   rainfall_historical: ['division_full', 'date', 'value'],
#   temperature_historical: ['division_full', 'date', 'value']
# };

payload = {
    "overwrite": True,
    "data": [
      get_statewide_stats("rainfall", 2026, 3)
    ]
}

response = requests.post(url, json=payload, headers=headers)

# 4. Print the result to see if it worked
print(f"Status Code: {response.status_code}")
print(f"Response Text: {response.text}")


--- Processing statewide (rainfall) ---
Returning stats for 2026-03: ['Statewide', 'Statewide', 'Statewide', '2026-03', np.float64(22.289707995859303), np.float64(15.436666994520824), np.float64(225.25280370430974), np.float64(2.0), np.float64(206.9470762479052)]
Status Code: 200
Response Text: {"modified":1}
